# Organization distribution

In [1]:
# Some imports
from cryptography.hazmat.backends import default_backend
from cryptography.x509.oid import ExtensionOID
from cryptography.x509.oid import NameOID
from cryptography import x509
import collections
import ipaddress
import pyasn
import radix
import json
import gzip

## Datasets

### Certificates

In [2]:
# Load certificates
certs = list()
with open("../data/certificates/certificates.json", "r") as f:
    for line in f:
        cert_str = json.loads(line)
        pem_bytes = cert_str.encode('utf-8')
        cert = x509.load_pem_x509_certificate(pem_bytes, default_backend())
        certs.append(cert)

### Organization mappings

In [3]:
# We download the deanonymized dataset of IP/ASN organizations from Geoff Huston
# https://www.potaroo.net/bgp/stats/nro/delegated-nro-extended-org

# Example ASN entries:
# - arin|US|asn|11|1|19840704|assigned|2c8438ff5e93d4a88cf7217faf2d6b5c|e-stats|Harvard University
# - apnic|TW|asn|1659|1|20020801|assigned|A91BDB29|e-stats|Taiwan Network Information Center
# - ripencc|EU|asn|261|1|19930901|assigned|abdb62a1-0a27-4c88-9d2d-0a90dcc0ea54|e-stats|Renater
# - afrinic|ZA|asn|1232|1|19910301|assigned|F36B9F4B|e-stats|TENET (The UNINET Project)
# - lacnic|CL|asn|1296|1|19910611|assigned|11510|e-stats|University of Chili

# Example IP entries:
# - apnic|CN|ipv4|103.26.156.0|1024|20130620|assigned|A929721E|e-stats|Beijing ChengQiTong Technology Co.,Ltd.
# - ripencc|GB|ipv4|31.192.96.0|2048|20110805|assigned|815e6132-68f3-4872-b6bf-cd12f414b92e|e-stats|Wavenet Limited

# The above file provides start address + count, so we need to convert it
def v4_range_to_cidr(ip, count):
    start = ipaddress.ip_address(ip)
    end = start + count - 1
    return list(ipaddress.summarize_address_range(start, end))

# Networks to orgs
net_v4_to_org, net_v6_to_org = radix.Radix(), radix.Radix()
# ASNs to orgs
asn_to_org = dict()

with gzip.open("../data/rir/nro-deanonymized-20260428.txt.gz", "rt") as f:
    for line in f:
        entry = line.strip().split("|")
        if entry[0] in {"arin", "apnic", "ripencc", "afrinic", "lacnic"}:
            if entry[2] == "ipv4":
                networks = v4_range_to_cidr(ip=entry[3],count=int(entry[4]))
                for network in networks:
                    radix_node = net_v4_to_org.add(str(network))
                    radix_node.data["org-cc"] = (entry[9],entry[1])
                    radix_node.data["rir"] = (entry[0])
            elif entry[2] == "ipv6":
                network = ipaddress.ip_network(f"{entry[3]}/{entry[4]}")
                radix_node = net_v6_to_org.add(str(network))
                radix_node.data["org-cc"] = (entry[9],entry[1])
                radix_node.data["rir"] = (entry[0])
            elif entry[2] == "asn":
                asn_to_org[int(entry[3])] = [(entry[9],entry[1]),entry[0]]

In [4]:
# This file contains AS names not present in the above file
with gzip.open("../data/rir/asn-not-in-nro.json.gz","rt") as f:
    asns = json.load(f)
    for asn in asns:
        org,cc = asns[asn][0].split("|")
        asn_to_org[int(asn)] = [(org,cc),asns[asn][1]]

### BGP routing table

In [5]:
# We downloaded the RouteViews dataset and parsed it
asndb = pyasn.pyasn("../data/routeviews/asndb_20260428.1200.gz")

### SAN IP addresses

In [6]:
# Get all IP addresses
ips = list()

# Get unique IP addresses before and after Let's Encrypt
ips_before, ips_after = set(), set()
for cert in certs:
    # Get SAN
    san = cert.extensions.get_extension_for_oid(ExtensionOID.SUBJECT_ALTERNATIVE_NAME).value
    for identifier in san:
        if isinstance(identifier,x509.IPAddress):
            # Store all IPs
            ip = str(identifier.value)
            ips.append(ip)
            # Store before and after Let's encrypt IPs
            if cert.not_valid_before_utc.month in {1,2,3,4,5,6}:
                ips_before.add(ip)
            else:
                ips_after.add(ip)

# Unique IPs
ips_unique = set(ips)
# Non-global IPs
ips_non_global = [i for i in ips_unique if not ipaddress.ip_address(i).is_global]
            

In [7]:
print(f"{len(certs):,} certificates:")
print(f"  {len(ips):,} IP addresses:")
print(f"    {len(ips_unique):,} unique IP addresses:")
print(f"      {len(ips_non_global):,} non-routable IP addresses")

517,619 certificates:
  582,753 IP addresses:
    227,854 unique IP addresses:
      0 non-routable IP addresses


## IP organizations

### Organization distribution (unique IPs)

In [8]:
ips_org_country = list()
orgs_unique, cc_unique = set(), set()

for ip in ips_unique:
    # If IPv4
    if ":" not in ip:
        org_cc = net_v4_to_org.search_best(ip).data["org-cc"]
    # If IPv6
    else:
        org_cc = net_v6_to_org.search_best(ip).data["org-cc"]
    # Organization/CC tuple
    ips_org_country.append(org_cc)
    orgs_unique.add(org_cc[0])
    cc_unique.add(org_cc[1])

ips_org_country_count = collections.Counter(ips_org_country).most_common()

In [9]:
print(f"There are {len(ips_org_country_count):,} ORG/CC pairs for {len(ips_unique):,} unique IPs")
print(f"  There are {len(orgs_unique):,} unique organizations")
print(f"  There are {len(cc_unique):,} unique countries")
print(f"---") 
print(f"The most common unique address organizations are:")
for entry,count in ips_org_country_count[:20]:
    print(f"  {entry[0]} ({entry[1]}): {count:,}")

There are 2,168 ORG/CC pairs for 227,854 unique IPs
  There are 2,137 unique organizations
  There are 123 unique countries
---
The most common unique address organizations are:
  Amazon Data Services Northern Virginia (US): 55,098
  Microsoft Corporation (US): 20,954
  Amazon.com, Inc. (US): 18,023
  Cloud Innovation Ltd (SC): 14,345
  Cogent Communications, LLC (US): 11,144
  DigitalOcean, LLC (US): 10,761
  Tencent Cloud Computing (Beijing) Co., Ltd (CN): 7,327
  Data Communication Business Group, (TW): 6,090
  CloudRadium L.L.C (US): 5,871
  Akamai Technologies, Inc. (US): 4,298
  TELUS Communications Inc. (CA): 4,072
  Tencent cloud computing (Beijing) Co., Ltd. (CN): 3,834
  Aliyun Computing Co., LTD (CN): 3,661
  Google LLC (US): 3,377
  RACKIP CONSULTANCY PTE. LTD. (SG): 2,977
  Microsoft Limited (GB): 2,927
  Netsec Limited (HK): 2,815
  The Constant Company, LLC (US): 2,689
  PEG TECH INC (US): 2,303
  METEVERSE LIMITED (CA): 2,230


In [10]:
# The above as Latex table
counter = 1
for entry,count in ips_org_country_count[:20]:
    print(f"{counter}. & {entry[0]} & {entry[1]} & {count:,} \\\\")
    counter += 1

1. & Amazon Data Services Northern Virginia & US & 55,098 \\
2. & Microsoft Corporation & US & 20,954 \\
3. & Amazon.com, Inc. & US & 18,023 \\
4. & Cloud Innovation Ltd & SC & 14,345 \\
5. & Cogent Communications, LLC & US & 11,144 \\
6. & DigitalOcean, LLC & US & 10,761 \\
7. & Tencent Cloud Computing (Beijing) Co., Ltd & CN & 7,327 \\
8. & Data Communication Business Group, & TW & 6,090 \\
9. & CloudRadium L.L.C & US & 5,871 \\
10. & Akamai Technologies, Inc. & US & 4,298 \\
11. & TELUS Communications Inc. & CA & 4,072 \\
12. & Tencent cloud computing (Beijing) Co., Ltd. & CN & 3,834 \\
13. & Aliyun Computing Co., LTD & CN & 3,661 \\
14. & Google LLC & US & 3,377 \\
15. & RACKIP CONSULTANCY PTE. LTD. & SG & 2,977 \\
16. & Microsoft Limited & GB & 2,927 \\
17. & Netsec Limited & HK & 2,815 \\
18. & The Constant Company, LLC & US & 2,689 \\
19. & PEG TECH INC & US & 2,303 \\
20. & METEVERSE LIMITED & CA & 2,230 \\


### Organization distribution (all IPs)

In [11]:
ips_org_country_all = list()
cloudflare_all = list()

for ip in ips:
    # If IPv4
    if ":" not in ip:
        org_cc = net_v4_to_org.search_best(ip).data["org-cc"]
    # If IPv6
    else:
        org_cc = net_v6_to_org.search_best(ip).data["org-cc"]
    # Organization/CC tuple
    ips_org_country_all.append(org_cc)
    # Cloudflare
    if org_cc == ("Cloudflare, Inc.", "US"):
        cloudflare_all.append(ip)

ips_org_country_all_count = collections.Counter(ips_org_country_all).most_common()
print(f"The most common all address organizations are:")
for entry,count in ips_org_country_all_count[:20]:
    print(f"  {entry[0]} ({entry[1]}): {count:,}")

The most common all address organizations are:
  Cloudflare, Inc. (US): 176,909
  Amazon Data Services Northern Virginia (US): 59,649
  Cloud Innovation Ltd (SC): 33,803
  Microsoft Corporation (US): 26,317
  Reliance Jio Infocomm Limited (IN): 25,402
  Tencent Cloud Computing (Beijing) Co., Ltd (CN): 25,018
  CloudRadium L.L.C (US): 24,212
  TELUS Communications Inc. (CA): 21,127
  Amazon.com, Inc. (US): 19,804
  DigitalOcean, LLC (US): 16,234
  Cogent Communications, LLC (US): 13,818
  Data Communication Business Group, (TW): 12,211
  Tencent cloud computing (Beijing) Co., Ltd. (CN): 8,808
  Google LLC (US): 6,481
  Akamai Technologies, Inc. (US): 5,261
  Brander Group Inc. (US): 5,243
  Aliyun Computing Co., LTD (CN): 5,066
  PEG TECH INC (US): 4,173
  Beijing Founder Broadband Network Technology Co.,Ltd (CN): 3,879
  Netsec Limited (HK): 3,749


In [12]:
# The above as Latex table
counter = 1
for entry,count in ips_org_country_all_count[:20]:
    print(f"{counter}. & {entry[0]} & {entry[1]} & {count:,} \\\\")
    counter += 1

1. & Cloudflare, Inc. & US & 176,909 \\
2. & Amazon Data Services Northern Virginia & US & 59,649 \\
3. & Cloud Innovation Ltd & SC & 33,803 \\
4. & Microsoft Corporation & US & 26,317 \\
5. & Reliance Jio Infocomm Limited & IN & 25,402 \\
6. & Tencent Cloud Computing (Beijing) Co., Ltd & CN & 25,018 \\
7. & CloudRadium L.L.C & US & 24,212 \\
8. & TELUS Communications Inc. & CA & 21,127 \\
9. & Amazon.com, Inc. & US & 19,804 \\
10. & DigitalOcean, LLC & US & 16,234 \\
11. & Cogent Communications, LLC & US & 13,818 \\
12. & Data Communication Business Group, & TW & 12,211 \\
13. & Tencent cloud computing (Beijing) Co., Ltd. & CN & 8,808 \\
14. & Google LLC & US & 6,481 \\
15. & Akamai Technologies, Inc. & US & 5,261 \\
16. & Brander Group Inc. & US & 5,243 \\
17. & Aliyun Computing Co., LTD & CN & 5,066 \\
18. & PEG TECH INC & US & 4,173 \\
19. & Beijing Founder Broadband Network Technology Co.,Ltd & CN & 3,879 \\
20. & Netsec Limited & HK & 3,749 \\


### Organizations before Let's Encrypt

In [13]:
ips_org_country_before = list()
orgs_unique_before, cc_unique_before = set(), set()

for ip in ips_before:
    # If IPv4
    if ":" not in ip:
        org_cc = net_v4_to_org.search_best(ip).data["org-cc"]
    # If IPv6
    else:
        org_cc = net_v6_to_org.search_best(ip).data["org-cc"]
    # Organization/CC tuple
    ips_org_country_before.append(org_cc)
    orgs_unique_before.add(org_cc[0])
    cc_unique_before.add(org_cc[1])


print(f"There were {len(set(ips_org_country_before)):,} ORG/CC pairs for {len(ips_before):,} unique IPs before July 1, 2025")
print(f"  There were {len(orgs_unique_before):,} unique organizations")
print(f"  There were {len(cc_unique_before):,} unique countries")

There were 1,565 ORG/CC pairs for 134,655 unique IPs before July 1, 2025
  There were 1,542 unique organizations
  There were 110 unique countries


### Cloudflare case study

In [14]:
# Cloudflare case study
cloudflare_count = collections.Counter(cloudflare_all)
cloudflare_rank = ips_org_country_count.index([i for i in ips_org_country_count if i[0][0] == "Cloudflare, Inc."][0]) + 1
print(f"Cloudflare ranks {cloudflare_rank}th with {len(cloudflare_count)} unique IPs")
print(f"Most common unique Cloudflare IPs are:")
for ip,count in cloudflare_count.most_common(n=5):
    print(f"  {ip}: {count:,}")

Cloudflare ranks 219th with 26 unique IPs
Most common unique Cloudflare IPs are:
  172.65.247.74: 88,597
  172.65.189.205: 88,268
  172.65.166.99: 3
  172.65.170.137: 3
  172.65.170.61: 3


In [15]:
# The two most common are 172.65.247.74 and 172.65.189.205
cloudflare_suspicious_ips = {"172.65.247.74", "172.65.189.205"}
cloudflare_suspicious_issuers = set()
cloudflare_suspicious_validity = list()

for cert in certs:
    # Get Issuer
    issuer_org = None
    for attribute in cert.issuer:
        if attribute.oid == NameOID.ORGANIZATION_NAME:
            issuer_org = attribute.value
    # Get SAN
    san = cert.extensions.get_extension_for_oid(ExtensionOID.SUBJECT_ALTERNATIVE_NAME).value
    for identifier in san:
        if isinstance(identifier,x509.IPAddress):
            ip = str(identifier.value)
            if ip in cloudflare_suspicious_ips:
                cloudflare_suspicious_issuers.add(issuer_org)
                cloudflare_suspicious_validity.append(cert.not_valid_before_utc)

delta = max(cloudflare_suspicious_validity)-min(cloudflare_suspicious_validity)
print(f"The CAs are: {cloudflare_suspicious_issuers}")
print(f"The {len(cloudflare_suspicious_validity):,} certificates were issued between {min(cloudflare_suspicious_validity)} and {max(cloudflare_suspicious_validity)}")
print(f"They were issued at the rate of {len(cloudflare_suspicious_validity)/delta.days} certificates per day")

The CAs are: {"Let's Encrypt"}
The 176,865 certificates were issued between 2025-07-02 17:53:36+00:00 and 2025-12-31 23:59:17+00:00
They were issued at the rate of 971.7857142857143 certificates per day


## IP countries

In [16]:
countries_all_count = collections.Counter([i[1] for i in ips_org_country_all]).most_common()
countries_unique_count = collections.Counter([i[1] for i in ips_org_country]).most_common()

print(f"The top countries of all the IPs and unique ones:")
for cc_all, cc_unique in zip(countries_all_count[:10],countries_unique_count[:10]):
    print(f"  {cc_all[0]}: {cc_all[1]:,} ({round(cc_all[1]*100/len(ips),2)}%) \t {cc_unique[0]}: {cc_unique[1]:,} ({round(cc_unique[1]*100/len(ips_unique),2)}%)")

The top countries of all the IPs and unique ones:
  US: 384,440 (65.97%) 	 US: 146,812 (64.43%)
  CN: 51,355 (8.81%) 	 CN: 21,026 (9.23%)
  SC: 33,972 (5.83%) 	 SC: 14,492 (6.36%)
  CA: 29,697 (5.1%) 	 CA: 8,689 (3.81%)
  IN: 25,743 (4.42%) 	 SG: 7,269 (3.19%)
  TW: 12,434 (2.13%) 	 TW: 6,213 (2.73%)
  SG: 9,466 (1.62%) 	 HK: 5,810 (2.55%)
  HK: 7,843 (1.35%) 	 GB: 4,052 (1.78%)
  GB: 5,283 (0.91%) 	 DE: 1,560 (0.68%)
  DE: 3,617 (0.62%) 	 JP: 1,340 (0.59%)


In [17]:
# The above as latex table
counter = 1
for cc_all, cc_unique in zip(countries_all_count[:10],countries_unique_count[:10]):
    print(f"{counter}. & {cc_all[0]} & {cc_all[1]:,} & {round(cc_all[1]*100/len(ips),2)}\% & {cc_unique[0]} & {cc_unique[1]:,} & {round(cc_unique[1]*100/len(ips_unique),2)}\% \\\\")
    counter += 1

1. & US & 384,440 & 65.97\% & US & 146,812 & 64.43\% \\
2. & CN & 51,355 & 8.81\% & CN & 21,026 & 9.23\% \\
3. & SC & 33,972 & 5.83\% & SC & 14,492 & 6.36\% \\
4. & CA & 29,697 & 5.1\% & CA & 8,689 & 3.81\% \\
5. & IN & 25,743 & 4.42\% & SG & 7,269 & 3.19\% \\
6. & TW & 12,434 & 2.13\% & TW & 6,213 & 2.73\% \\
7. & SG & 9,466 & 1.62\% & HK & 5,810 & 2.55\% \\
8. & HK & 7,843 & 1.35\% & GB & 4,052 & 1.78\% \\
9. & GB & 5,283 & 0.91\% & DE & 1,560 & 0.68\% \\
10. & DE & 3,617 & 0.62\% & JP & 1,340 & 0.59\% \\


## IP RIRs

In [18]:
rir_all = list()
for ip in ips:
    # If IPv4
    if ":" not in ip:
        rir = net_v4_to_org.search_best(ip).data["rir"]
    # If IPv6
    else:
        rir = net_v6_to_org.search_best(ip).data["rir"]
    # Store the RIR
    rir_all.append(rir)

rir_unique = list()
for ip in ips_unique:
    # If IPv4
    if ":" not in ip:
        rir = net_v4_to_org.search_best(ip).data["rir"]
    # If IPv6
    else:
        rir = net_v6_to_org.search_best(ip).data["rir"]
    # Store the RIR
    rir_unique.append(rir)

rir_all_count = collections.Counter(rir_all).most_common()
rir_unique_count = collections.Counter(rir_unique).most_common()

for rir_1,rir_2 in zip(rir_all_count,rir_unique_count):
    print(f"{rir_1[0].upper()}: {rir_1[1]:,} ({round(rir_1[1]*100/len(rir_all),2)}%) \t {rir_2[0].upper()}: {rir_2[1]:,} ({round(rir_2[1]*100/len(rir_unique),2)}%)")

ARIN: 389,630 (66.86%) 	 ARIN: 139,703 (61.31%)
APNIC: 120,278 (20.64%) 	 APNIC: 52,522 (23.05%)
RIPENCC: 36,929 (6.34%) 	 RIPENCC: 19,578 (8.59%)
AFRINIC: 35,671 (6.12%) 	 AFRINIC: 15,878 (6.97%)
LACNIC: 245 (0.04%) 	 LACNIC: 173 (0.08%)


In [19]:
# The above data in Latex format
for rir_1,rir_2 in zip(rir_all_count,rir_unique_count):
    print(f"{rir_1[0].upper()} & {rir_1[1]:,} & {round(rir_1[1]*100/len(rir_all),2)}\% & {rir_2[0].upper()} & {rir_2[1]:,} & {round(rir_2[1]*100/len(rir_unique),2)}\% \\\\")

ARIN & 389,630 & 66.86\% & ARIN & 139,703 & 61.31\% \\
APNIC & 120,278 & 20.64\% & APNIC & 52,522 & 23.05\% \\
RIPENCC & 36,929 & 6.34\% & RIPENCC & 19,578 & 8.59\% \\
AFRINIC & 35,671 & 6.12\% & AFRINIC & 15,878 & 6.97\% \\
LACNIC & 245 & 0.04\% & LACNIC & 173 & 0.08\% \\


## ASN organizations

In [20]:
# List ASNs not found in the RIR archive and the JSON file
asn_not_found = set()

for ip in ips_unique:
    asn, prefix = asndb.lookup(ip)
    if asn:
        if asn not in asn_to_org:
            asn_not_found.add(asn)

if asn_not_found:
    print(f"We did not find {len(asn_not_found)} ASNs in the RIR dataset ({asn_not_found})")
    print(f"Please fill in the asn-not-in-nro.json.gz file before moving forward")

### Organization distribution (unique IPs)

In [21]:
# List of ASN (org,cc) tuples
asns_org_country = list()
orgs_unique, cc_unique = set(), set()

for ip in ips_unique:
    asn, prefix = asndb.lookup(ip)
    if asn:
        org_cc = asn_to_org[asn][0]
        # If no ORG name available use AS number
        if org_cc[0]:
            orgs_unique.add(org_cc[0])
        else:
            orgs_unique.add(f"AS{asn}")
            org_cc = (f"AS{asn}",org_cc[1])

        asns_org_country.append(org_cc)
        cc_unique.add(org_cc[1])
        
asns_org_country_count = collections.Counter(asns_org_country).most_common()

In [22]:
print(f"There are {len(asns_org_country_count):,} ASN ORG/CC pairs for {len(ips_unique):,} unique IPs")
print(f"  There are {len(orgs_unique):,} unique organizations")
print(f"  There are {len(cc_unique):,} unique countries")
print(f"---") 
print(f"The most common unique address organizations are:")
for entry,count in asns_org_country_count[:20]:
    print(f"  {entry[0]} ({entry[1]}): {count:,}")

There are 1,640 ASN ORG/CC pairs for 227,854 unique IPs
  There are 1,635 unique organizations
  There are 124 unique countries
---
The most common unique address organizations are:
  Amazon.com, Inc. (US): 73,573
  Microsoft Corporation (US): 24,053
  Shenzhen Tencent Computer Systems Company Limited (CN): 15,116
  CNSERVERS LLC (US): 12,601
  DigitalOcean, LLC (US): 12,019
  AS3462 (TW): 6,090
  Akamai Technologies, Inc. (SG): 6,004
  Hangzhou Alibaba Advertising Co.,Ltd. (CN): 5,698
  MULTACOM CORPORATION (US): 5,593
  POWER LINE (HK) CO., LIMITED (HK): 5,190
  PEG TECH INC (US): 4,830
  Nebula Global LLC (US): 4,226
  Meteverse Limited. (CA): 3,804
  The Constant Company, LLC (US): 3,757
  Google LLC (US): 3,378
  CTG Server Limited (HK): 3,300
  Netsec Limited (HK): 3,050
  Alibaba (US) Technology Co., Ltd. (US): 1,877
  AS138415 (HK): 1,512
  OVH SAS (FR): 1,459


In [23]:
# The above as Latex table
counter = 1
for entry,count in asns_org_country_count[:20]:
    print(f"{counter}. & {entry[0]} & {entry[1]} & {count:,} \\\\")
    counter += 1

1. & Amazon.com, Inc. & US & 73,573 \\
2. & Microsoft Corporation & US & 24,053 \\
3. & Shenzhen Tencent Computer Systems Company Limited & CN & 15,116 \\
4. & CNSERVERS LLC & US & 12,601 \\
5. & DigitalOcean, LLC & US & 12,019 \\
6. & AS3462 & TW & 6,090 \\
7. & Akamai Technologies, Inc. & SG & 6,004 \\
8. & Hangzhou Alibaba Advertising Co.,Ltd. & CN & 5,698 \\
9. & MULTACOM CORPORATION & US & 5,593 \\
10. & POWER LINE (HK) CO., LIMITED & HK & 5,190 \\
11. & PEG TECH INC & US & 4,830 \\
12. & Nebula Global LLC & US & 4,226 \\
13. & Meteverse Limited. & CA & 3,804 \\
14. & The Constant Company, LLC & US & 3,757 \\
15. & Google LLC & US & 3,378 \\
16. & CTG Server Limited & HK & 3,300 \\
17. & Netsec Limited & HK & 3,050 \\
18. & Alibaba (US) Technology Co., Ltd. & US & 1,877 \\
19. & AS138415 & HK & 1,512 \\
20. & OVH SAS & FR & 1,459 \\


### Organization distribution (all IPs)

In [24]:
# List of ASN (org,cc) tuples
asns_org_country_all = list()

for ip in ips:
    asn, prefix = asndb.lookup(ip)
    if asn:
        org_cc = asn_to_org[asn][0]
        # If no ORG name available use AS number
        if not org_cc[0]:
            org_cc = (f"AS{asn}",org_cc[1])
        asns_org_country_all.append(org_cc)
        
asns_org_country_all_count = collections.Counter(asns_org_country_all).most_common()

print(f"The most common all ASN organizations are:")
for entry,count in asns_org_country_all_count[:20]:
    print(f"  {entry[0]} ({entry[1]}): {count:,}")

The most common all ASN organizations are:
  Cloudflare, Inc. (US): 176,927
  Amazon.com, Inc. (US): 79,991
  CNSERVERS LLC (US): 49,158
  Shenzhen Tencent Computer Systems Company Limited (CN): 42,839
  Microsoft Corporation (US): 29,901
  MULTACOM CORPORATION (US): 28,776
  Reliance Jio Infocomm Limited (IN): 25,402
  DigitalOcean, LLC (US): 18,218
  AS3462 (TW): 12,211
  Hangzhou Alibaba Advertising Co.,Ltd. (CN): 7,903
  POWER LINE (HK) CO., LIMITED (HK): 7,669
  Akamai Technologies, Inc. (SG): 7,380
  PEG TECH INC (US): 7,064
  Google LLC (US): 6,483
  OVH SAS (FR): 5,441
  Nebula Global LLC (US): 4,908
  The Constant Company, LLC (US): 4,406
  Meteverse Limited. (CA): 4,222
  Netsec Limited (HK): 4,072
  CTG Server Limited (HK): 3,760


In [25]:
# The above as Latex table
counter = 1
for entry,count in asns_org_country_all_count[:20]:
    print(f"{counter}. & {entry[0]} & {entry[1]} & {count:,} \\\\")
    counter += 1

1. & Cloudflare, Inc. & US & 176,927 \\
2. & Amazon.com, Inc. & US & 79,991 \\
3. & CNSERVERS LLC & US & 49,158 \\
4. & Shenzhen Tencent Computer Systems Company Limited & CN & 42,839 \\
5. & Microsoft Corporation & US & 29,901 \\
6. & MULTACOM CORPORATION & US & 28,776 \\
7. & Reliance Jio Infocomm Limited & IN & 25,402 \\
8. & DigitalOcean, LLC & US & 18,218 \\
9. & AS3462 & TW & 12,211 \\
10. & Hangzhou Alibaba Advertising Co.,Ltd. & CN & 7,903 \\
11. & POWER LINE (HK) CO., LIMITED & HK & 7,669 \\
12. & Akamai Technologies, Inc. & SG & 7,380 \\
13. & PEG TECH INC & US & 7,064 \\
14. & Google LLC & US & 6,483 \\
15. & OVH SAS & FR & 5,441 \\
16. & Nebula Global LLC & US & 4,908 \\
17. & The Constant Company, LLC & US & 4,406 \\
18. & Meteverse Limited. & CA & 4,222 \\
19. & Netsec Limited & HK & 4,072 \\
20. & CTG Server Limited & HK & 3,760 \\


### Cloud Innovation case study

In [26]:
# Cloud Innovation is registered in Africa but is not routed by Africa

cloud_innovation_asns = set()
cloud_innovation_asns_orgs = list()
cloud_innovation_asns_countries = set()
cloud_innovation_asns_rir = list()

for ip in ips_unique:
    # Get the organization
    if ":" not in ip:
        org = net_v4_to_org.search_best(ip).data["org-cc"][0]
    else:
        org = net_v6_to_org.search_best(ip).data["org-cc"][0]
    # Only keep Cloud Innovation-owned IPs
    if org == "Cloud Innovation Ltd":
        # Now check who is routing this IP
        asn, prefix = asndb.lookup(ip)
        if asn:
            cloud_innovation_asns.add(asn)
            asn_org,asn_cc = asn_to_org[asn][0]
            cloud_innovation_asns_orgs.append(asn_org)
            cloud_innovation_asns_countries.add(asn_cc)
            cloud_innovation_asns_rir.append(asn_to_org[asn][1])

print(f"Cloud Innovation IPs are routed by {len(cloud_innovation_asns)} ASNs ({len(set(cloud_innovation_asns_orgs))} organizations from {len(cloud_innovation_asns_countries)} countries)")
print(f"")
print(f"RIR distribution of routing autonomous systems is:")
for rir,count in collections.Counter(cloud_innovation_asns_rir).most_common():
    print(f"  {rir.upper()}: {count:,}")

print(f"The most common: {collections.Counter(cloud_innovation_asns_orgs).most_common(n=5)}")

Cloud Innovation IPs are routed by 66 ASNs (60 organizations from 14 countries)

RIR distribution of routing autonomous systems is:
  ARIN: 8,232
  APNIC: 6,045
  RIPENCC: 48
  LACNIC: 2
  AFRINIC: 1
The most common: [('CNSERVERS LLC', 6383), ('POWER LINE (HK) CO., LIMITED', 2657), ('', 978), ('AROSSCLOUD INC.', 662), ('Turing Group Limited', 595)]


## ASN countries

In [27]:
countries_asn_all_count = collections.Counter([i[1] for i in asns_org_country_all]).most_common()
countries_asn_unique_count = collections.Counter([i[1] for i in asns_org_country]).most_common()

print(f"The top ASN countries of all the IPs and unique ones:")
for cc_all, cc_unique in zip(countries_asn_all_count[:10],countries_asn_unique_count[:10]):
    print(f"  {cc_all[0]}: {cc_all[1]:,} ({round(cc_all[1]*100/len(ips),2)}%) \t {cc_unique[0]}: {cc_unique[1]:,} ({round(cc_unique[1]*100/len(ips_unique),2)}%)")

The top ASN countries of all the IPs and unique ones:
  US: 426,588 (73.2%) 	 US: 156,172 (68.54%)
  CN: 53,718 (9.22%) 	 CN: 22,955 (10.07%)
  HK: 25,737 (4.42%) 	 HK: 18,764 (8.24%)
  IN: 25,714 (4.41%) 	 SG: 6,541 (2.87%)
  TW: 12,313 (2.11%) 	 TW: 6,161 (2.7%)
  SG: 8,067 (1.38%) 	 CA: 4,514 (1.98%)
  FR: 6,218 (1.07%) 	 FR: 1,706 (0.75%)
  CA: 5,325 (0.91%) 	 DE: 1,251 (0.55%)
  DE: 3,010 (0.52%) 	 GB: 1,122 (0.49%)
  GB: 2,603 (0.45%) 	 NZ: 744 (0.33%)


In [28]:
# The above as latex table
counter = 1
for cc_all, cc_unique in zip(countries_asn_all_count[:10],countries_asn_unique_count[:10]):
    print(f"{counter}. & {cc_all[0]} & {cc_all[1]:,} & {round(cc_all[1]*100/len(ips),2)}\% & {cc_unique[0]} & {cc_unique[1]:,} & {round(cc_unique[1]*100/len(ips_unique),2)}\% \\\\")
    counter += 1

1. & US & 426,588 & 73.2\% & US & 156,172 & 68.54\% \\
2. & CN & 53,718 & 9.22\% & CN & 22,955 & 10.07\% \\
3. & HK & 25,737 & 4.42\% & HK & 18,764 & 8.24\% \\
4. & IN & 25,714 & 4.41\% & SG & 6,541 & 2.87\% \\
5. & TW & 12,313 & 2.11\% & TW & 6,161 & 2.7\% \\
6. & SG & 8,067 & 1.38\% & CA & 4,514 & 1.98\% \\
7. & FR & 6,218 & 1.07\% & FR & 1,706 & 0.75\% \\
8. & CA & 5,325 & 0.91\% & DE & 1,251 & 0.55\% \\
9. & DE & 3,010 & 0.52\% & GB & 1,122 & 0.49\% \\
10. & GB & 2,603 & 0.45\% & NZ & 744 & 0.33\% \\


## ASN RIRs

In [29]:
rir_asn_all = list()
for ip in ips:
    asn, prefix = asndb.lookup(ip)
    if asn:
        rir_asn_all.append(asn_to_org[asn][1])

rir_asn_unique = list()
for ip in ips_unique:
    asn, prefix = asndb.lookup(ip)
    if asn:
        rir_asn_unique.append(asn_to_org[asn][1])

rir_asn_all_count = collections.Counter(rir_asn_all).most_common()
rir_asn_unique_count = collections.Counter(rir_asn_unique).most_common()

for rir_1,rir_2 in zip(rir_asn_all_count,rir_asn_unique_count):
    print(f"{rir_1[0].upper()}: {rir_1[1]:,} ({round(rir_1[1]*100/len(rir_all),2)}%) \t {rir_2[0].upper()}: {rir_2[1]:,} ({round(rir_2[1]*100/len(rir_unique),2)}%)")

ARIN: 430,454 (73.87%) 	 ARIN: 159,542 (70.02%)
APNIC: 129,250 (22.18%) 	 APNIC: 57,607 (25.28%)
RIPENCC: 21,103 (3.62%) 	 RIPENCC: 9,308 (4.09%)
LACNIC: 260 (0.04%) 	 LACNIC: 192 (0.08%)
AFRINIC: 90 (0.02%) 	 AFRINIC: 76 (0.03%)


In [30]:
# The above data in Latex format
for rir_1,rir_2 in zip(rir_asn_all_count,rir_asn_unique_count):
    print(f"{rir_1[0].upper()} & {rir_1[1]:,} & {round(rir_1[1]*100/len(rir_all),2)}\% & {rir_2[0].upper()} & {rir_2[1]:,} & {round(rir_2[1]*100/len(rir_unique),2)}\% \\\\")

ARIN & 430,454 & 73.87\% & ARIN & 159,542 & 70.02\% \\
APNIC & 129,250 & 22.18\% & APNIC & 57,607 & 25.28\% \\
RIPENCC & 21,103 & 3.62\% & RIPENCC & 9,308 & 4.09\% \\
LACNIC & 260 & 0.04\% & LACNIC & 192 & 0.08\% \\
AFRINIC & 90 & 0.02\% & AFRINIC & 76 & 0.03\% \\
